# 007 — Train Item2Vec (MovieLens, Ray Tune + MLflow + Evidently)

Full MLOps port of the reference `src/model_item2vec/main.py`:

- **Ray Train + Ray Tune** — HP search over `embedding_dim` (choice `[64, 256]`),
  `learning_rate` + `l2_reg` (loguniform), `num_samples` trials via `TorchTrainer` +
  `Tuner`. Best trial (min `val_loss`) → a final `TorchTrainer` run with the best
  params.
- **MLflow tracking** — each trial logs to experiment `item2vec/hyperparameter_tuning`,
  the final run to `item2vec/final_model`. The final run logs `idm.json` + the
  TorchScript SkipGram to the **Model Registry** (`item2vec_skipgram`) and tags the
  best `val_loss` version as `champion`.
- **Evidently** — a classification report (`ClassificationPreset`) is generated at
  the end of each fit and logged to MLflow (`val_roc_auc` + precision/recall).

Config is env-driven (`configs/item2vec.yaml` reads `${MLFLOW_TRACKING_URI}`,
`${RAY_ADDRESS}`, …) so the same notebook runs:

- **local-pro** — `RAY_ADDRESS=local`, MLflow via `docker compose up -d`
  (UI at http://localhost:5000, artifact store = MinIO `s3://recsys-ops/mlflow/`).
- **AWS real** — `RAY_ADDRESS=ray://...` (EKS KubeRay), MLflow on EC2 + RDS + S3.

Inputs live under `feature/output/engineer/` (produced by the feature engineering
phase). See `docs/README.md` "Path D" for the MLflow + Ray setup.

In [ ]:
import os
from pathlib import Path

from loguru import logger

# Project root = two levels up from this notebook (models/item2vec/).
ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
print("project root:", ROOT)
print("sequences present:", (ROOT / "feature/output/engineer/train_item_sequence.jsonl").exists())
print("overfit batch present:", (ROOT / "feature/output/engineer/batch_sequences_overfit.jsonl").exists())
print("idm present:", (ROOT / "feature/output/engineer/idm.json").exists())


## Run the training

`train.py` is a CLI entrypoint. Run it as a module from the project root so
the `models.item2vec` / `feature.id_mapper` package imports resolve.

Prerequisites (local-pro):

1. Start the MLflow stack: `docker compose up -d` → MLflow UI at
   http://localhost:5000 (backend = Postgres `mlflow` DB, artifacts = MinIO
   `s3://recsys-ops/mlflow/`).
2. Export the `.env` so `train.py` picks up `MLFLOW_TRACKING_URI`,
   `MLFLOW_S3_ENDPOINT_URL`, `MLFLOW_AWS_*`, `RAY_ADDRESS`:

```bash
export $(grep -v '^#' .env | xargs)
uv run python -m models.item2vec.train --config configs/item2vec.yaml
```

Ray Tune runs `num_samples` trials (each logs to MLflow
`item2vec/hyperparameter_tuning`), picks the best by `val_loss`, then runs a
final training that logs the TorchScript model to the Registry
(`item2vec_skipgram`) and tags the champion version. Add `--overfit` to run a
single-batch sanity check first (off by default).

In [ ]:
# Load .env (MLFLOW_*, RAY_ADDRESS) into this shell, then run Ray Tune + final training.
!set -a && . ./.env && set +a && uv run python -m models.item2vec.train --config configs/item2vec.yaml

## Inspect the result

The final checkpoint + an `idm.json` copy are written to
`models/output/item2vec/final_model/`. The TorchScript model + `idm.json` are
also in the MLflow Model Registry under `item2vec_skipgram`, with the champion
version tagged. Open http://localhost:5000 to browse experiments + the
registry, or query it programmatically below.

In [ ]:
import os

import torch
from feature.id_mapper import IDMapper
from models.item2vec.model import SkipGram

final_dir = ROOT / "models/output/item2vec/final_model"
ckpt_path = final_dir / "best-checkpoint.ckpt"
print("checkpoint exists:", ckpt_path.exists())

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu")
    hparams = ckpt.get("hyper_parameters", {})
    print("hparams keys:", list(hparams.keys()))

    # Rebuild the model from the checkpoint weights and look up one embedding.
    idm = IDMapper().load(str(final_dir / "idm.json"))
    model = SkipGram(num_items=len(idm.item_to_index), embedding_dim=hparams.get("embedding_dim", 64))
    print("vocab size:", len(idm.item_to_index))
    print("sample item_id -> idx:", list(idm.item_to_index.items())[:3])

# Query the MLflow Model Registry for the champion version of item2vec_skipgram.
try:
    import mlflow

    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
    client = mlflow.tracking.MlflowClient()
    versions = client.search_model_versions("name='item2vec_skipgram'")
    print(f"\nMLflow registry 'item2vec_skipgram': {len(versions)} version(s)")
    for v in versions:
        champion = client.get_model_version_tag("item2vec_skipgram", v.version, "champion")
        run = client.get_run(v.run_id)
        val_loss = run.data.metrics.get("final.val_loss") or run.data.metrics.get("tuning.val_loss")
        print(f"  v{v.version} | champion={champion} | val_loss={val_loss} | status={v.status}")
except Exception as e:
    print(f"\nMLflow inspect skipped: {e}")